In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import trange, tqdm
import copy

from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder, LabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, cohen_kappa_score, precision_score, recall_score, f1_score
from sklearn.utils import class_weight

import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.nn import Conv1d, LSTM, GRU
from torch.functional import F
from torch.utils.data import Dataset, TensorDataset, DataLoader
import pytorch_lightning
import torchmetrics
import torchsummary
from torch.nn import Sequential
from torch.nn.modules import Module
from torch.optim import Adam, Adamax, Adagrad, AdamW, Adafactor, Adadelta
from torch.optim.lr_scheduler import CosineAnnealingLR, ReduceLROnPlateau, ExponentialLR
#from torch.fft import fft, fftfreq, fftshift

from scipy.fft import fft

import skorch
from skorch import NeuralNetClassifier
from skorch.dataset import ValidSplit
from skorch.callbacks import EarlyStopping, Checkpoint, LRScheduler, EpochScoring, EpochTimer

f:\Conda\envs\DeepLearning-Torch\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(device)

cuda


# **Load Dataset**

In [3]:
X_windows = np.load("F:/Project-Motor Imagery Classification/BCICIV_2a_gdf/Tensor/X_alt.npy")
y_categorical = np.load("F:/Project-Motor Imagery Classification/BCICIV_2a_gdf/Tensor/y.npy")

In [4]:
X_windows = torch.FloatTensor(X_windows)
y_categorical = torch.FloatTensor(y_categorical)
y_encoded = y_integer = np.argmax(y_categorical, axis=1)

In [5]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_windows, y_categorical, test_size=0.2, random_state=42, stratify=y_categorical
)
print(f"Train: {X_train.shape}, {y_train.shape}")
print(f"Test: {X_test.shape}, {y_test.shape}")

Train: torch.Size([8089, 22, 2, 50]), torch.Size([8089, 4])
Test: torch.Size([2023, 22, 2, 50]), torch.Size([2023, 4])


In [6]:
# Create TensorDatasets
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

In [7]:
# Create DataLoaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
valid_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

In [8]:
# Iterate through training data
for batch_idx, (eeg_data, labels) in enumerate(train_loader):
    print(f"Batch {batch_idx}:")
    print(f"  EEG data shape: {eeg_data.shape}")  # [batch_size, channels, time_points]
    print(f"  Labels shape: {labels.shape}")      # [batch_size]
    
    # Your model training code goes here
    # predictions = model(eeg_data)
    # loss = criterion(predictions, labels)
    
    if batch_idx == 2:  # Show first 3 batches
        break

# Check one batch from each loader
train_batch = next(iter(train_loader))
val_batch = next(iter(valid_loader))

print(f"\nTrain batch: {train_batch[0].shape}, {train_batch[1].shape}")
print(f"Validation batch: {val_batch[0].shape}, {val_batch[1].shape}")

Batch 0:
  EEG data shape: torch.Size([32, 22, 2, 50])
  Labels shape: torch.Size([32, 4])
Batch 1:
  EEG data shape: torch.Size([32, 22, 2, 50])
  Labels shape: torch.Size([32, 4])
Batch 2:
  EEG data shape: torch.Size([32, 22, 2, 50])
  Labels shape: torch.Size([32, 4])

Train batch: torch.Size([32, 22, 2, 50]), torch.Size([32, 4])
Validation batch: torch.Size([32, 22, 2, 50]), torch.Size([32, 4])


In [9]:
X_train.shape

torch.Size([8089, 22, 2, 50])

In [10]:
# Define input shape from X_train
channels = X_train.shape[1]
timesteps = X_train.shape[2]       # 50
features = X_train.shape[1]        # 22 * 2 = 44
num_classes = y_train.shape[1]     # 4

In [11]:
X_train[0].shape

torch.Size([22, 2, 50])

In [12]:
X0 = X_train[0].unsqueeze(0)

In [13]:
X0 = X_train[:32]

In [14]:
X0.shape

torch.Size([32, 22, 2, 50])

In [15]:
X0[:,0,:].shape

torch.Size([32, 2, 50])

In [16]:
X0[:,0,:].permute(0,2,1).shape

torch.Size([32, 50, 2])

# **Testing Branched Architectures**

In [17]:
branched_input = nn.ModuleList([
    Sequential(
    #Depthwise Convolution    
    Conv1d(in_channels=2, out_channels=2, kernel_size=3, padding=0),
    #Pointwise Convolution
    Conv1d(in_channels=2, out_channels=32, kernel_size=1, padding=0)
) for i in range(0,22)
])

In [18]:
branched_input, branched_input[0]

(ModuleList(
   (0-21): 22 x Sequential(
     (0): Conv1d(2, 2, kernel_size=(3,), stride=(1,))
     (1): Conv1d(2, 32, kernel_size=(1,), stride=(1,))
   )
 ),
 Sequential(
   (0): Conv1d(2, 2, kernel_size=(3,), stride=(1,))
   (1): Conv1d(2, 32, kernel_size=(1,), stride=(1,))
 ))

In [19]:
branches = []
for i in range(0,22):
    branches.append(branched_input[i](X0[:,i,:]))
branched_concat = torch.cat(branches)

In [20]:
branched_concat.shape

torch.Size([704, 32, 48])

# **Testing Recurrent Architectures**

In [26]:
rnn_in = branched_concat.permute(0,2,1) #change to (batch_size, sample_length, number of samples) 

In [27]:
rnn_LSTM = LSTM(input_size=32,hidden_size=64, batch_first=True)
rnn_GRU = GRU(input_size=32, hidden_size=64, batch_first=True)

In [28]:
rnn_out = rnn_GRU(rnn_in)[0]

In [29]:
rnn_out.shape

torch.Size([704, 48, 64])

In [30]:
rnn_out[:,-1,:].shape

torch.Size([704, 64])

# **Testing ConvNext Architecture**

In [21]:
class LayerNorm1d(nn.Module):
    """LayerNorm over channel dimension for (B, C, L)"""
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.norm = nn.LayerNorm(dim, eps=eps)
    
    def forward(self, x):
        x = x.permute(0, 2, 1)  # (B, L, C)
        x = self.norm(x)
        return x.permute(0, 2, 1)  # (B, C, L)

class ConvNeXtBlock1D(nn.Module):
    def __init__(self, dim, drop_path=0.):
        super().__init__()
        # Depthwise conv (preserves channels)
        self.dwconv = nn.Conv1d(dim, dim, kernel_size=7, padding=3, groups=dim)
        # LayerNorm
        self.norm = LayerNorm1d(dim)
        # Pointwise expansion (dim → 4*dim)
        self.pwconv1 = nn.Linear(dim, 4 * dim)
        self.act = nn.GELU()
        # Pointwise projection (4*dim → dim)
        self.pwconv2 = nn.Linear(4 * dim, dim)

    def forward(self, x):
        input = x
        x = self.dwconv(x)           # (B, dim, L)
        x = self.norm(x)             # (B, dim, L)
        x = x.permute(0, 2, 1)       # (B, L, dim) for Linear
        x = self.pwconv1(x)          # (B, L, 4*dim)
        x = self.act(x)
        x = self.pwconv2(x)          # (B, L, dim)
        x = x.permute(0, 2, 1)       # (B, dim, L)
        x = input + x                # Residual connection
        return x

In [22]:
ConvNeXt = ConvNeXtBlock1D(dim=2)
ConvNeXt(X0[:,0,:]).shape

torch.Size([32, 2, 50])

# **Testing Branched ConvNext**

In [23]:
branched_input = nn.ModuleList([
    Sequential(
    ConvNeXtBlock1D(dim=2),
    #Depthwise Convolution    
    Conv1d(in_channels=2, out_channels=2, kernel_size=3, padding=0),
    #Pointwise Convolution
    Conv1d(in_channels=2, out_channels=32, kernel_size=1, padding=0)
) for i in range(0,22)
])

In [24]:
branched_input, branched_input[0]

(ModuleList(
   (0-21): 22 x Sequential(
     (0): ConvNeXtBlock1D(
       (dwconv): Conv1d(2, 2, kernel_size=(7,), stride=(1,), padding=(3,), groups=2)
       (norm): LayerNorm1d(
         (norm): LayerNorm((2,), eps=1e-06, elementwise_affine=True)
       )
       (pwconv1): Linear(in_features=2, out_features=8, bias=True)
       (act): GELU(approximate='none')
       (pwconv2): Linear(in_features=8, out_features=2, bias=True)
     )
     (1): Conv1d(2, 2, kernel_size=(3,), stride=(1,))
     (2): Conv1d(2, 32, kernel_size=(1,), stride=(1,))
   )
 ),
 Sequential(
   (0): ConvNeXtBlock1D(
     (dwconv): Conv1d(2, 2, kernel_size=(7,), stride=(1,), padding=(3,), groups=2)
     (norm): LayerNorm1d(
       (norm): LayerNorm((2,), eps=1e-06, elementwise_affine=True)
     )
     (pwconv1): Linear(in_features=2, out_features=8, bias=True)
     (act): GELU(approximate='none')
     (pwconv2): Linear(in_features=8, out_features=2, bias=True)
   )
   (1): Conv1d(2, 2, kernel_size=(3,), stride=(1,)

In [25]:
branches = []
for i in range(0,22):
    branches.append(branched_input[i](X0[:,i,:]))
branched_concat = torch.cat(branches)

In [26]:
branched_concat.shape

torch.Size([704, 32, 48])

# **Putting it together**

Lightweight cheapstake

In [37]:
class Final_Arch(nn.Module):
    def __init__(self, input_channel=2, num_channels=22):
        super().__init__()
        self.num_channels = num_channels
        self.branched_input = nn.ModuleList([
            Sequential(
            ConvNeXtBlock1D(dim=input_channel),
            #Depthwise Convolution    
            Conv1d(in_channels=2, out_channels=2, kernel_size=3, padding=0),
            #Pointwise Convolution
            Conv1d(in_channels=2, out_channels=32, kernel_size=1, padding=0)
        ) for i in range(0,22)
        ])
        self.GRU = GRU(input_size=32, hidden_size=64, batch_first=True)
        
    def forward(self, x):
        branch_in = []
        for i in range(0,self.num_channels):
            branch_in.append(self.branched_input[i](x[:,i,:]))
        x = torch.cat(branch_in)
        print(x.shape)
        x, _ = self.GRU(x.permute(0,2,1))
        print(x.shape)
        #x = x[:,-1,:]
        #print(x.shape)
        return x

In [38]:
model = Final_Arch()

In [39]:
model

Final_Arch(
  (branched_input): ModuleList(
    (0-21): 22 x Sequential(
      (0): ConvNeXtBlock1D(
        (dwconv): Conv1d(2, 2, kernel_size=(7,), stride=(1,), padding=(3,), groups=2)
        (norm): LayerNorm1d(
          (norm): LayerNorm((2,), eps=1e-06, elementwise_affine=True)
        )
        (pwconv1): Linear(in_features=2, out_features=8, bias=True)
        (act): GELU(approximate='none')
        (pwconv2): Linear(in_features=8, out_features=2, bias=True)
      )
      (1): Conv1d(2, 2, kernel_size=(3,), stride=(1,))
      (2): Conv1d(2, 32, kernel_size=(1,), stride=(1,))
    )
  )
  (GRU): GRU(32, 64, batch_first=True)
)

In [40]:
x = model(X0)

torch.Size([704, 32, 48])
torch.Size([704, 48, 64])


In [41]:
x.permute(2,1,0).shape

torch.Size([64, 48, 704])

In [42]:
X0.shape

torch.Size([32, 22, 2, 50])

EEGXception Channelwise input

In [68]:
class XceptionBock(nn.Module):
    def __init__(self, in_channel, out_channel):
        super().__init__()

        self.residual = nn.Conv1d(in_channels=in_channel, out_channels=out_channel, kernel_size=1, padding='same')

        self.main_path = Sequential(
            nn.ReLU(),
            nn.Conv1d(in_channels=in_channel, out_channels=out_channel, kernel_size=8, padding='same', groups=in_channel),
            nn.Conv1d(in_channels=out_channel, out_channels=out_channel, kernel_size=1),
            nn.BatchNorm1d(out_channel),

            nn.ReLU(),
            nn.Conv1d(in_channels=out_channel, out_channels=out_channel, kernel_size=8, padding='same', groups=out_channel),
            nn.Conv1d(in_channels=out_channel, out_channels=out_channel, kernel_size=1),
            nn.BatchNorm1d(out_channel)
        )

        #addition of residual and main tensor in forward pass
        
        self.pool = nn.MaxPool1d(2)
    def forward(self, x):
        res = self.residual(x)
        x = self.main_path(x)
        x = x + res
        x = self.pool(x)

        return x

In [ ]:
class LayerNorm1d(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.norm = nn.LayerNorm(dim, eps=eps)
    def forward(self, x):
        x = x.permute(0, 2, 1)  # (B, L, C)
        x = self.norm(x)
        return x.permute(0, 2, 1)  # (B, C, L)

class ConvNeXtBlock1D(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dwconv = nn.Conv1d(dim, dim, kernel_size=7, padding=3, groups=dim)
        self.norm = LayerNorm1d(dim)
        self.pwconv1 = nn.Conv1d(dim, 4 * dim, kernel_size=1)
        self.act = nn.GELU()
        self.pwconv2 = nn.Conv1d(4 * dim, dim, kernel_size=1)
    def forward(self, x):
        residual = x
        x = self.dwconv(x)
        x = self.norm(x)
        x = self.pwconv1(x)
        x = self.act(x)
        x = self.pwconv2(x)
        return x + residual

class EEGXceptionNet(nn.Module):
    def __init__(self, input_channel=2, num_classes=4, name="EEGXceptionNet"):
        super().__init__()
        self.name = name
        self.num_channels = 22  # ✅ FIXED: now defined!

        # One branch per EEG channel
        self.branched_input = nn.ModuleList([
            nn.Sequential(
                ConvNeXtBlock1D(dim=input_channel),
                nn.Conv1d(input_channel, 32, kernel_size=1)  # Project to 32
            ) for _ in range(self.num_channels)
        ])

        # Shared backbone
        self.input_block = nn.Sequential(
            nn.Conv1d(32, 64, kernel_size=8, padding='same'),
            nn.BatchNorm1d(64),
            nn.ReLU()
        )

        self.branch_convs = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(64, 32, kernel_size=k, padding='same', groups=32),
                nn.Conv1d(32, 32, kernel_size=1),
                nn.BatchNorm1d(32),
                nn.ReLU(inplace=True)
            ) for k in [3, 5, 8, 11]
        ])

        self.post_concat = nn.Sequential(
            nn.Conv1d(128, 128, kernel_size=1, padding='same'),
            nn.BatchNorm1d(128),
            nn.ReLU()
        )

        self.xception1 = XceptionBock(128, 128)
        self.xception2 = XceptionBock(128, 256)

        self.squeeze_and_excitation = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(256, 256 // 8),
            nn.ReLU(),
            nn.Linear(256 // 8, 256),
            nn.Sigmoid()
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )
        self.Soft=nn.Softmax(dim=1)

    def forward(self, x):
        """
        x: (B, 22, 2, 50)
        """
        B, C, F, T = x.shape
        assert C == self.num_channels and F == 2

        # Process each EEG channel independently
        features = []
        for i in range(self.num_channels):
            xi = x[:, i, :, :]  # (B, 2, 50)
            fi = self.branched_input[i](xi)  # (B, 32, ~50)
            features.append(fi)

        x = torch.stack(features, dim=1)  # (B, 22, 32, L)
        x = x.mean(dim=1)  # (B, 32, L)

        x = self.input_block(x)
        branches = [branch(x) for branch in self.branch_convs]
        x = torch.cat(branches, dim=1)
        x = self.post_concat(x)
        x = self.xception1(x)
        x = self.xception2(x)

        # Squeeze-and-Excitation
        se_weights = self.squeeze_and_excitation(x)  # (B, 256)
        se_weights = se_weights.unsqueeze(-1)  # (B, 256, 1)
        x = x * se_weights

        x = self.classifier(x)
        x = self.Soft(x)
        return x

In [77]:
model = EEGXceptionNet(input_channel=2, num_classes=4)

In [78]:
x = model(X0)

In [79]:
x.shape

torch.Size([32, 4])